# **05 - SelfCheck-Prompt** 

In [ ]:
# Restart after running this cell.
!pip install -q "transformers==4.36.2" selfcheckgpt sentencepiece "accelerate>=0.26,<1.0" "bitsandbytes>=0.43.0"
!python -m spacy download en_core_web_sm -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to 

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/selfcheckgpt_replication_results"
except ImportError:                      # running outside Colab
    RESULTS_DIR = "./selfcheckgpt_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["SC_RESULTS_DIR"] = RESULTS_DIR
if RESULTS_DIR not in sys.path: sys.path.insert(0, RESULTS_DIR)
try:
    from sc_common import *            # shared config, dataset, metrics, cache helpers 
except ModuleNotFoundError:
    raise FileNotFoundError(f"Upload sc_common.py into {RESULTS_DIR} first.")
describe_device(); ram("after setup")

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/evaluation-00000-of-00001-e91191b8f(…):   0%|          | 0.00/2.56M [00:00<?, ?B/s]

Generating evaluation split:   0%|          | 0/238 [00:00<?, ? examples/s]

Using device: cuda
Tesla T4 - 14.6 GB
Results / caches -> /content/drive/MyDrive/selfcheckgpt_replication_results
[RAM after setup] this process 0.8 GB | system used 1.6 / 12.7 GB


In [ ]:
PROMPT_TEMPLATE = "Context: {context}\n\nSentence: {sentence}\n\nIs the sentence supported by the context above? Answer Yes or No.\n\nAnswer: "
NA_OUTPUTS = Counter()

def map_answer(text):
    # same mapping as the package: Yes -> 0.0, No -> 1.0, everything else (N/A) -> 0.5
    t = text.lower().strip()
    if t[:3] == "yes": return 0.0
    if t[:2] == "no":  return 1.0
    NA_OUTPUTS[t] += 1
    return 0.5

if LLM_BACKEND == "hf":
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    llm_tok = AutoTokenizer.from_pretrained(HF_LLM, use_fast=False)
    llm_tok.padding_side = "left"
    if llm_tok.pad_token is None: llm_tok.pad_token = llm_tok.eos_token
    use_4bit = (GPU_GB < 20) if HF_4BIT == "auto" else bool(HF_4BIT)
    if use_4bit:
        llm = AutoModelForCausalLM.from_pretrained(HF_LLM, device_map={"": 0}, low_cpu_mem_usage=True,
                quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
    else:
        llm = AutoModelForCausalLM.from_pretrained(HF_LLM, torch_dtype=torch.float16, device_map={"": 0}, low_cpu_mem_usage=True)
    llm.eval()
    print(f"{HF_LLM} loaded ({'4-bit NF4' if use_4bit else 'fp16'})")

    @torch.no_grad()
    def prompt_matrix(sentences, samples):
        M = np.zeros((len(sentences), len(samples)))
        for si, sentence in enumerate(sentences):
            prompts = [PROMPT_TEMPLATE.format(context=s.replace("\n", " "), sentence=sentence) for s in samples]   
            for b in range(0, len(prompts), PROMPT_BATCH):
                chunk = prompts[b:b + PROMPT_BATCH]
                enc = llm_tok(chunk, return_tensors="pt", padding=True).to(device)
                out = llm.generate(**enc, max_new_tokens=5, do_sample=False, pad_token_id=llm_tok.pad_token_id)
                texts = llm_tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
                for k, t in enumerate(texts): M[si, b + k] = map_answer(t)
        return M

elif LLM_BACKEND == "openai":
    from concurrent.futures import ThreadPoolExecutor
    from selfcheckgpt.modeling_selfcheck_apiprompt import SelfCheckAPIPrompt
    assert os.environ.get("OPENAI_API_KEY"), "set os.environ['OPENAI_API_KEY'] first"
    api = SelfCheckAPIPrompt(client_type="openai", model=OPENAI_MODEL)

    def _ask(prompt, retries=5):
        for k in range(retries):
            try: return api.completion(prompt)
            except Exception as e:
                time.sleep(2 ** k)
        return ""                                  # counted as N/A (0.5)

    def prompt_matrix(sentences, samples):
        prompts = [PROMPT_TEMPLATE.format(context=s.replace("\n", " "), sentence=sent) for sent in sentences for s in samples]
        with ThreadPoolExecutor(OPENAI_WORKERS) as ex: texts = list(ex.map(_ask, prompts))
        return np.array([map_answer(t) for t in texts]).reshape(len(sentences), len(samples))
ram("after loading LLM")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

mistralai/Mistral-7B-Instruct-v0.2 loaded (4-bit NF4)
[RAM after loading LLM] this process 2.1 GB | system used 2.7 / 12.7 GB


In [4]:
if VERIFY_AGAINST_PACKAGE and LLM_BACKEND == "hf":
    # batched, left-padded generation must give the same Yes/No decisions as one-prompt-at-a-time generation
    r = rows[PROMPT_IDS[0]]; sents, samp = r["gpt3_sentences"][:2], r["gpt3_text_samples"][:3]
    single = np.zeros((len(sents), len(samp)))
    for a_, sent in enumerate(sents):
        for b_, s_ in enumerate(samp):
            enc = llm_tok(PROMPT_TEMPLATE.format(context=s_.replace("\n", " "), sentence=sent), return_tensors="pt").to(device)
            g = llm.generate(**enc, max_new_tokens=5, do_sample=False, pad_token_id=llm_tok.pad_token_id)
            single[a_, b_] = map_answer(llm_tok.decode(g[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
    batched = prompt_matrix(sents, samp)
    print(f"batched vs single-prompt decisions agree on {int((single == batched).sum())}/{single.size} (sentence, sample) pairs")
    NA_OUTPUTS.clear()

t0 = time.time()
MATS["prompt"]["samples"] = run_cached(f"prompt_{PROMPT_SLUG}_samples", lambda i: prompt_matrix(rows[i]["gpt3_sentences"], get_samples(i)), PROMPT_IDS, f"SelfCheck-Prompt ({PROMPT_NAME})", save_every=2)
print(f"done in {(time.time() - t0) / 60:.1f} min;  non Yes/No outputs (-> 0.5): {sum(NA_OUTPUTS.values())}  {dict(NA_OUTPUTS.most_common(5))}")
register_main(f"SelfCheck-Prompt ({PROMPT_NAME})", scores_from_matrices("prompt", MATS["prompt"]["samples"]))

if RUN_WIKIBIO_EVIDENCE:
    MATS["prompt"]["wiki"] = run_cached(f"prompt_{PROMPT_SLUG}_wiki", lambda i: prompt_matrix(rows[i]["gpt3_sentences"], get_samples(i, "wiki")), PROMPT_IDS, "Prompt (WikiBio)", save_every=10)
    RES_WIKI[f"WikiBio+Prompt ({PROMPT_NAME})"] = evaluate(scores_from_matrices("prompt", MATS["prompt"]["wiki"]))


batched vs single-prompt decisions agree on 6/6 (sentence, sample) pairs
SelfCheck-Prompt (Mistral-7B-Instruct-v0.2): 0 cached, 30 to compute


SelfCheck-Prompt (Mistral-7B-Instruct-v0.2):   0%|          | 0/30 [00:00<?, ?it/s]

done in 28.5 min;  non Yes/No outputs (-> 0.5): 24  {'partly.': 9, 'the context above': 5, 'the sentence is': 4, 'partially.': 2, 'marshall man': 1}
SelfCheck-Prompt (Mistral-7B-Instruct-v0.2) {'NonFact': np.float64(92.19), 'NonFact*': np.float64(46.76), 'Factual': np.float64(63.68), 'Pearson': np.float64(77.95), 'Spearman': np.float64(76.11), 'n_passages': 30}
Prompt (WikiBio): 0 cached, 30 to compute


Prompt (WikiBio):   0%|          | 0/30 [00:00<?, ?it/s]